## Intro to Provisioning with Docker Compose

In addition to the FL Simulator and POC modes, NVFlare ships with a provisioning system that can be used to generate scripts and configuration files for secure deployment of the FL server and clients.  These configurations also include certificates for establishing identity and secure communication between participants, and are packaged in unique subdirectory for each client and server.  We call these packages "startup kits".

To get started with the provisioning system, we can look at the `nvflare provision` command.


In [1]:
!nvflare provision -h

usage: nvflare provision [-h] [-p PROJECT_FILE] [-w WORKSPACE]
                         [-c CUSTOM_FOLDER] [--add_user ADD_USER]
                         [--add_client ADD_CLIENT]

optional arguments:
  -h, --help            show this help message and exit
  -p PROJECT_FILE, --project_file PROJECT_FILE
                        file to describe FL project
  -w WORKSPACE, --workspace WORKSPACE
                        directory used by provision
  -c CUSTOM_FOLDER, --custom_folder CUSTOM_FOLDER
                        additional folder to load python codes
  --add_user ADD_USER   yaml file for added user
  --add_client ADD_CLIENT
                        yaml file for added client


Provisioning is based on a PROJECT_FILE that defines the server and clients, and the set of NVFlare Builder modules that are used to generate the startup kits.  You can run `nvflare provision` without any arguments to generate a sample `project.yml` file.  This will prompt you to choose either a High-Availability (HA) or non-HA configuration.


### Provisioning for Docker Compose deployment

For this demo, we've already generated a `project.yml` config, choosing non-HA for simplicity, and have added an additional Builder module to generate a Docker compose environment for deploying locally.  This confiuguration is in [`project-non-HA.yml`](project-non-HA.yml) (click to open in an editor).  The DockerBuilder module was added following the WorkspaceBuilder config with:
```yaml
  - path: nvflare.lighter.impl.docker.DockerBuilder
    args:
      base_image: gtc-dli-nvflare-monai:latest
```

We can generate the project workspace by running `nvflare provision -p project-non-HA.yml`.  In this example, we'll set up `monai_workspace` for the next notebook that introduces MONAI FL integration.


In [2]:
!nvflare provision -w monai_workspace -p files/project-non-HA.yml


Path list (sys.path) for python codes loading: ['/opt/conda/bin', '/opt/conda/lib/python38.zip', '/opt/conda/lib/python3.8', '/opt/conda/lib/python3.8/lib-dynload', '/opt/conda/lib/python3.8/site-packages', '/opt/conda/lib/python3.8/site-packages/torchtext-0.11.0a0-py3.8-linux-x86_64.egg', '/opt/conda/lib/python3.8/site-packages/certifi-2022.9.14-py3.8.egg', '/opt/conda/lib/python3.8/site-packages/functorch-0.3.0a0-py3.8-linux-x86_64.egg', '/flare/NVFlare', '/flare/notebooks', '/flare/notebooks/.'] 

Project yaml file: /flare/notebooks/project-non-HA.yml.
Generated results can be found under /flare/notebooks/monai_workspace/monai_demo/prod_00.  Builder's wip folder removed.


**_Note:_** Because we're running in a container, we need make a few adjustments to the Docker compose config generated during provision, and then copy the workspace to a location accessible by the host system.  *This is not necessary when running on a local system.*

In [3]:
# Fix the Docker compose setup for running within the container

%env HOST_PATH=/tmp/monai_workspace/monai_demo/prod_00
!echo HOST_PATH=/tmp/monai_workspace/monai_demo/prod_00 >> monai_workspace/monai_demo/prod_00/.env
!sed -i s,nvflare-service,flare-lab-client,g monai_workspace/monai_demo/prod_00/.env
!sed -i s,/usr/local/bin/python3,/opt/conda/bin/python3,g monai_workspace/monai_demo/prod_00/.env
!sed -i s,./server1,\$\{HOST_PATH\}/server1,g monai_workspace/monai_demo/prod_00/compose.yaml
!sed -i s,./site-1,\$\{HOST_PATH\}/site-1,g monai_workspace/monai_demo/prod_00/compose.yaml
!sed -i s,./site-2,\$\{HOST_PATH\}/site-2,g monai_workspace/monai_demo/prod_00/compose.yaml
!cp -rf monai_workspace /tmp/monai_workspace

env: HOST_PATH=/tmp/monai_workspace/monai_demo/prod_00


### Launching the FL system
Now that we have our provisioning `workspace` directory with the `monai_demo/prod_00` configuration, we can launch the FL server and clients each in a docker container by using the `compose.yml` and `.env` that were generated during provisioning.

In [14]:
!docker compose --file monai_workspace/monai_demo/prod_00/compose.yaml --env-file monai_workspace/monai_demo/prod_00/.env up -d

[+] Running 0/0
 ⠋ Network prod_00_default  Creating                                       0.1s
[+] Running 1/1
 ⠿ Network prod_00_default  Created                                        0.1s
 ⠋ Container site-1         Creating                                       0.1s
 ⠋ Container site-2         Creating                                       0.1s
 ⠋ Container server1        Creating                                       0.1s
[+] Running 1/4
 ⠿ Network prod_00_default  Created                                        0.1s
 ⠙ Container site-1         Creating                                       0.2s
 ⠙ Container site-2         Creating                                       0.2s
 ⠙ Container server1        Creating                                       0.2s
[+] Running 1/4
 ⠿ Network prod_00_default  Created                                        0.1s
 ⠿ Container site-1         Starting                                       0.3s
 ⠿ Container site-2         Starting                    

We can verify that the `server1` and two clients, `site-1` and `site-2` are each running in a container instance (we will also see a fourth container `nvflare-monai` that's hosting this notebook and DLI content.)

In [5]:
!docker ps

CONTAINER ID   IMAGE                   COMMAND                  CREATED         STATUS         PORTS                                                                               NAMES
b5dd54cca3dc   gtc-dli-nvflare-monai   "/opt/docker/entrypo…"   6 seconds ago   Up 5 seconds   6006/tcp, 8888/tcp                                                                  site-2
bc441efc2b99   gtc-dli-nvflare-monai   "/opt/docker/entrypo…"   6 seconds ago   Up 5 seconds   6006/tcp, 8888/tcp, 0.0.0.0:8002-8003->8002-8003/tcp, :::8002-8003->8002-8003/tcp   server1
e32a9f960c6f   gtc-dli-nvflare-monai   "/opt/docker/entrypo…"   6 seconds ago   Up 5 seconds   6006/tcp, 8888/tcp                                                                  site-1
dde9889173f2   gtc-dli-nvflare-monai   "/opt/docker/entrypo…"   2 minutes ago   Up 2 minutes                                                                                       nvflare-monai


### Connecting to the Docker Compose deployment
Even though we're running locally via Docker compose, the server and all clients are running securely in their own container instance.  Compared to POC mode, where we were running without authentication enabled, we will need to use the `admin@nvidia.com` toolkit to establish a secure session, providing the admin username `admin@nvidia.com` and the certificates provided in the `monai_workspace/monai_demo/prod_00/admin@nvidia.com` directory.

In [15]:
admin_dir = "monai_workspace/monai_demo/prod_00/admin@nvidia.com"
admin_user = "admin@nvidia.com"
from nvflare.fuel.flare_api.flare_api import new_secure_session

admin_session = new_secure_session(
    username = admin_user,
    startup_kit_location = admin_dir
)
print(admin_session.get_system_info())

SystemInfo
server_info:
status: stopped, start_time: Wed Mar  8 23:28:24 2023
client_info:
site-2(last_connect_time: Wed Mar  8 23:29:06 2023)
site-1(last_connect_time: Wed Mar  8 23:29:06 2023)
job_info:

